# Portfolio follow-up checks

Extra checks run after the main analysis (`EVs_Analysis.ipynb`) for the portfolio write-up at
[portfolio-mo.vercel.app/projects/us-ev-market-share](https://portfolio-mo.vercel.app/projects/us-ev-market-share).

In [1]:
import pandas as pd

ev = pd.read_csv('../data/cleaned-vehicle-registrations.csv')
total = ev['Total_Vehicles'].sum()
total_ev = ev['Electric (EV)'].sum()
national = total_ev / total * 100
ev['ev_pct'] = ev['Electric (EV)'] / ev['Total_Vehicles'] * 100
print(f'National EV share: {national:.2f}% ({total_ev:,.0f} of {total:,.0f} vehicles)')

National EV share: 1.24% (3,555,900 of 287,096,500 vehicles)


## 1. How concentrated are EVs in California?

In [2]:
ca = ev.set_index('State').loc['California']
outside = (total_ev - ca['Electric (EV)']) / (total - ca['Total_Vehicles']) * 100
lowest = ev.loc[ev['ev_pct'].idxmin()]

print(f"California's share of U.S. vehicles: {ca['Total_Vehicles'] / total * 100:.1f}%")
print(f"California's share of U.S. EVs:      {ca['Electric (EV)'] / total_ev * 100:.1f}%")
print(f'EV share outside California:         {outside:.2f}%')
print(f"California vs {lowest['State']}: {ca['ev_pct']:.2f}% vs {lowest['ev_pct']:.2f}% = {ca['ev_pct'] / lowest['ev_pct']:.0f}x")

California's share of U.S. vehicles: 12.8%
California's share of U.S. EVs:      35.3%
EV share outside California:         0.92%
California vs North Dakota: 3.41% vs 0.13% = 27x


## 2. Ranking states for infrastructure investment

Investment reaches the most potential drivers where a state has many vehicles and a below-average EV share. One number captures both: how many more EVs the state would need to reach the national average share.

`EVs short = total vehicles × national share − current EVs`

In [3]:
ev['evs_short_of_avg'] = ev['Total_Vehicles'] * national / 100 - ev['Electric (EV)']
ranking = (
    ev.sort_values('evs_short_of_avg', ascending=False)
    .reset_index(drop=True)
    .assign(rank=lambda d: d.index + 1)
    [['rank', 'State', 'Total_Vehicles', 'ev_pct', 'evs_short_of_avg']]
    .round({'ev_pct': 2, 'evs_short_of_avg': 0})
)
ranking.head(10)

,rank,State,Total_Vehicles,ev_pct,evs_short_of_avg
0,1,Texas,25796600.0,0.89,89410.0
1,2,Ohio,10317300.0,0.49,77387.0
2,3,Pennsylvania,10211000.0,0.69,56271.0
3,4,Michigan,8525400.0,0.59,55293.0
4,5,Indiana,6172100.0,0.42,50346.0
5,6,Tennessee,6538800.0,0.51,47788.0
6,7,Alabama,4835900.0,0.27,46896.0
7,8,Wisconsin,5529000.0,0.45,43581.0
8,9,Missouri,5626000.0,0.48,42782.0
9,10,North Carolina,9085500.0,0.77,42331.0


In [4]:
# Where the original report's picks land on this measure
ranking[ranking['State'].isin(['Texas', 'Florida', 'Georgia'])]

,rank,State,Total_Vehicles,ev_pct,evs_short_of_avg
0,1,Texas,25796600.0,0.89,89410.0
16,17,Georgia,9642400.0,0.96,27028.0
47,48,Florida,18583200.0,1.37,-24733.0


Texas, Ohio, and Pennsylvania rank first to third. Florida has the third-largest fleet but is already above the national average (a negative gap). Georgia ranks 17th.

## 3. A note on flex-fuel (E85)

Flex-fuel vehicles are about 7% of the fleet, but the column counts vehicles that *can* run on E85. Most fill up with regular gasoline, so this share overstates real ethanol use.

In [5]:
ev['Ethanol/Flex (E85)'].sum() / total * 100

np.float64(7.050103362458268)